# Project 5 — Quantization and Efficiency Pareto Frontier

Compresses the best CNN from [Project 1](01_emg_gesture_classification.ipynb), entirely in software — no hardware purchase required.

## Configurations swept
FP32 baseline, FP16, post-training dynamic int8, post-training static int8 (with a calibration set), quantization-aware training (QAT), structured pruning at 30/50/70% sparsity, and knowledge distillation from the CNN into a small MLP.

## Tooling
`torch.ao.quantization`, ONNX Runtime, TFLite converter (optional), `ptflops`/`fvcore` for MAC counts.

## Metrics per configuration
macro F1, model size (KB), inference latency (median + p95 over 1000 runs, pinned to a single CPU thread — a microcontroller is single-threaded, so multi-threaded timings are meaningless here), and MACs.

## Presentation
Pareto plots (accuracy vs. size, accuracy vs. latency) with the knee identified, plus a per-layer sensitivity analysis (quantize one layer at a time) showing which layers must stay in float.

**The finding to watch for**: LDA on hand-crafted features (Project 1) is frequently within 2–3 points of the CNN at a tiny fraction of the compute. If that's what you find here, lead with it — reporting that the fancy model wasn't worth it demonstrates more judgment than a top-line accuracy number.

**Free C/C++ credential**: export the int8 model to ONNX and drive it from a small C++ harness via the ONNX Runtime C++ API (or hand-write the quantized conv loop in C and assert it matches the Python output within tolerance).


## 1. Setup

In [ ]:
# Colab setup
# !pip install -q torch onnx onnxruntime ptflops scikit-learn numpy pandas matplotlib

import copy
import time
import io
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.ao.quantization as tq
from torch.utils.data import DataLoader

try:
    from ptflops import get_model_complexity_info
    HAS_PTFLOPS = True
except ImportError:
    HAS_PTFLOPS = False
    print("ptflops not installed — pip install ptflops for MAC counts")

try:
    import onnx
    import onnxruntime as ort
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False
    print("onnx / onnxruntime not installed — ONNX export section will be skipped")

RNG_SEED = 0
torch.manual_seed(RNG_SEED)
np.random.seed(RNG_SEED)

torch.set_num_threads(1)  # single-thread timing, matches a microcontroller

try:
    CFG
    EMG1DCNN
    WindowDataset
    files
except NameError as e:
    raise RuntimeError(
        "Run 01_emg_gesture_classification.ipynb first (or %run it) so CFG, "
        "EMG1DCNN, WindowDataset, files, base CNN, and eval data are in scope."
    ) from e


## 2. Load the Project 1 CNN + held-out eval set

Assumes `base_model` (a trained `EMG1DCNN`), `mu`/`sd` z-score stats, and windowed eval data exist from a prior run of Project 1 (or Project 4). If starting fresh in this notebook, re-run the within-subject split + `fit_cnn` from Project 1 first.

In [ ]:
if files:
    rec = prepare_subject(files[0], CFG)
    n_classes = CFG.n_classes_subset + 1

    Xw_all, yw_all, repw_all = make_windows(rec["emg"], rec["restimulus"], rec["rerepetition"], CFG)
    tr_mask, te_mask = split_within_subject(yw_all, repw_all, CFG)

    mu = Xw_all[tr_mask].mean(axis=(0, 1), keepdims=True)
    sd = Xw_all[tr_mask].std(axis=(0, 1), keepdims=True) + 1e-8
    Xtr = (Xw_all[tr_mask] - mu) / sd
    Xte = (Xw_all[te_mask] - mu) / sd
    ytr, yte = yw_all[tr_mask], yw_all[te_mask]

    fp32_model = fit_cnn(Xtr, ytr, n_classes, CFG, epochs=15)
    fp32_model.eval()

    calib_idx = np.random.RandomState(RNG_SEED).choice(len(Xtr), size=min(200, len(Xtr)), replace=False)
    X_calib = Xtr[calib_idx]
else:
    print("No Ninapro DB2 files found — populate CFG.data_dir, then re-run.")


## 3. Common eval harness: accuracy, size, latency, MACs

In [ ]:
def eval_macro_f1(model, X, y, device="cpu"):
    from sklearn.metrics import f1_score
    model = model.to(device).eval()
    with torch.no_grad():
        x = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1).to(device)
        out = model(x)
        pred = out.argmax(dim=1).cpu().numpy()
    return f1_score(y, pred, average="macro")


def model_size_kb(model) -> float:
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return len(buf.getvalue()) / 1024.0


def bench_latency_ms(model, sample_x, n_runs=1000, device="cpu"):
    """Median + p95 single-sample inference latency, single CPU thread."""
    model = model.to(device).eval()
    x = torch.tensor(sample_x[:1], dtype=torch.float32).permute(0, 2, 1).to(device)
    with torch.no_grad():
        for _ in range(20):  # warmup
            model(x)
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model(x)
            times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    return {"median_ms": float(np.median(times)), "p95_ms": float(np.percentile(times, 95))}


def count_macs(model, input_shape):
    if not HAS_PTFLOPS:
        return None
    macs, params = get_model_complexity_info(model, input_shape, as_strings=False,
                                               print_per_layer_stat=False, verbose=False)
    return macs


def collect_metrics(name, model, X_eval, y_eval, input_shape, extra=None):
    row = {"config": name,
           "macro_f1": eval_macro_f1(model, X_eval, y_eval),
           "size_kb": model_size_kb(model),
           **bench_latency_ms(model, X_eval)}
    macs = count_macs(model, input_shape)
    row["macs"] = macs
    if extra:
        row.update(extra)
    return row


## 4. FP32 baseline

In [ ]:
results = []
if files:
    input_shape = (CFG.n_channels, CFG.window_len)
    results.append(collect_metrics("fp32_baseline", fp32_model, Xte, yte, input_shape))
    display(pd.DataFrame(results))


## 5. FP16

In [ ]:
if files:
    fp16_model = copy.deepcopy(fp32_model).half()

    def eval_fp16(model, X, y):
        from sklearn.metrics import f1_score
        model.eval()
        with torch.no_grad():
            x = torch.tensor(X, dtype=torch.float16).permute(0, 2, 1)
            pred = model(x).float().argmax(dim=1).numpy()
        return f1_score(y, pred, average="macro")

    row = {"config": "fp16", "macro_f1": eval_fp16(fp16_model, Xte, yte),
           "size_kb": model_size_kb(fp16_model)}
    # latency bench for fp16 on CPU is often *slower* than fp32 (no native CPU fp16 kernels) —
    # report it anyway, the honesty of that result is part of the point.
    try:
        row.update(bench_latency_ms(fp16_model, Xte.astype(np.float16)))
    except Exception as e:
        row["median_ms"], row["p95_ms"] = None, None
        print(f"fp16 CPU latency bench skipped: {e}")
    results.append(row)
    display(pd.DataFrame(results))


## 6. Post-training dynamic int8

In [ ]:
if files:
    dynamic_int8_model = tq.quantize_dynamic(
        copy.deepcopy(fp32_model), {nn.Linear}, dtype=torch.qint8
    )
    # Note: dynamic quantization only touches nn.Linear by default; Conv1d layers stay fp32.
    # This under-states achievable compression vs. static/QAT below, which is realistic —
    # dynamic quantization is the "free lunch" tier, not the ceiling.
    results.append(collect_metrics("dynamic_int8", dynamic_int8_model, Xte, yte, input_shape))
    display(pd.DataFrame(results))


## 7. Post-training static int8 (with calibration)

In [ ]:
class QuantWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.quant = tq.QuantStub()
        self.model = model
        self.dequant = tq.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.model(x)
        return self.dequant(x)


if files:
    static_model = QuantWrapper(copy.deepcopy(fp32_model))
    static_model.eval()
    static_model.qconfig = tq.get_default_qconfig("fbgemm")
    tq.prepare(static_model, inplace=True)

    with torch.no_grad():
        x_calib = torch.tensor(X_calib, dtype=torch.float32).permute(0, 2, 1)
        static_model(x_calib)  # calibration pass — observes activation ranges

    static_int8_model = tq.convert(static_model, inplace=False)
    results.append(collect_metrics("static_int8_calibrated", static_int8_model, Xte, yte, input_shape))
    display(pd.DataFrame(results))


## 8. Quantization-aware training (QAT)

In [ ]:
if files:
    qat_model = QuantWrapper(copy.deepcopy(fp32_model))
    qat_model.train()
    qat_model.qconfig = tq.get_default_qat_qconfig("fbgemm")
    tq.prepare_qat(qat_model, inplace=True)

    opt = torch.optim.Adam(qat_model.parameters(), lr=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(WindowDataset(Xtr, ytr), batch_size=64, shuffle=True)

    for epoch in range(5):  # short fine-tune, QAT starts from a converged fp32 model
        for xb, yb in loader:
            opt.zero_grad()
            out = qat_model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()

    qat_model.eval()
    qat_int8_model = tq.convert(qat_model, inplace=False)
    results.append(collect_metrics("qat_int8", qat_int8_model, Xte, yte, input_shape))
    display(pd.DataFrame(results))


## 9. Structured pruning (30 / 50 / 70% sparsity)

In [ ]:
import torch.nn.utils.prune as prune

def prune_model(model, amount):
    m = copy.deepcopy(model)
    for module in m.modules():
        if isinstance(module, nn.Conv1d):
            prune.ln_structured(module, name="weight", amount=amount, n=2, dim=0)
            prune.remove(module, "weight")  # bake the mask in so state_dict reflects zeros
    return m


if files:
    for sparsity in (0.3, 0.5, 0.7):
        pruned = prune_model(fp32_model, sparsity)
        # brief fine-tune to recover accuracy after pruning
        opt = torch.optim.Adam(pruned.parameters(), lr=1e-4)
        loss_fn = nn.CrossEntropyLoss()
        loader = DataLoader(WindowDataset(Xtr, ytr), batch_size=64, shuffle=True)
        pruned.train()
        for epoch in range(5):
            for xb, yb in loader:
                opt.zero_grad()
                loss = loss_fn(pruned(xb), yb)
                loss.backward()
                opt.step()
        pruned.eval()
        results.append(collect_metrics(f"pruned_{int(sparsity*100)}pct", pruned, Xte, yte, input_shape,
                                        extra={"sparsity": sparsity}))
    display(pd.DataFrame(results))


## 10. Knowledge distillation into a small MLP

In [ ]:
class SmallMLP(nn.Module):
    """Operates on the same Hudgins+AR feature vector as Project 1's LDA/SVM arms —
    a fair 'small model' comparison point, not just a shrunk CNN."""
    def __init__(self, in_dim, n_classes, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def distill(teacher, X_windows_train, y_train, n_classes, cfg, epochs=20, T=2.0, alpha=0.5):
    F_train = extract_feature_matrix(X_windows_train, cfg.ar_order)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler().fit(F_train)
    F_train_s = scaler.transform(F_train).astype(np.float32)

    student = SmallMLP(F_train_s.shape[1], n_classes)
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    ce = nn.CrossEntropyLoss()

    teacher.eval()
    with torch.no_grad():
        x_teacher = torch.tensor(X_windows_train, dtype=torch.float32).permute(0, 2, 1)
        teacher_logits = teacher(x_teacher)
        teacher_soft = torch.softmax(teacher_logits / T, dim=1)

    y_t = torch.tensor(y_train, dtype=torch.long)
    x_t = torch.tensor(F_train_s, dtype=torch.float32)

    student.train()
    for epoch in range(epochs):
        opt.zero_grad()
        student_logits = student(x_t)
        hard_loss = ce(student_logits, y_t)
        soft_loss = nn.functional.kl_div(
            torch.log_softmax(student_logits / T, dim=1), teacher_soft, reduction="batchmean"
        ) * (T ** 2)
        loss = alpha * hard_loss + (1 - alpha) * soft_loss
        loss.backward()
        opt.step()
    return student, scaler


if files:
    student_model, feat_scaler = distill(fp32_model, Xtr, ytr, n_classes, CFG)

    F_test = extract_feature_matrix(Xte, CFG.ar_order)
    F_test_s = feat_scaler.transform(F_test).astype(np.float32)

    from sklearn.metrics import f1_score
    student_model.eval()
    with torch.no_grad():
        pred = student_model(torch.tensor(F_test_s)).argmax(dim=1).numpy()
    distill_f1 = f1_score(yte, pred, average="macro")

    row = {"config": "distilled_mlp", "macro_f1": distill_f1, "size_kb": model_size_kb(student_model)}
    x_bench = torch.tensor(F_test_s[:1])
    with torch.no_grad():
        for _ in range(20):
            student_model(x_bench)
        times = []
        for _ in range(1000):
            t0 = time.perf_counter()
            student_model(x_bench)
            times.append((time.perf_counter() - t0) * 1000)
    row["median_ms"] = float(np.median(times))
    row["p95_ms"] = float(np.percentile(times, 95))
    results.append(row)
    display(pd.DataFrame(results))


## 11. Baseline for comparison: LDA on hand-crafted features (from Project 1)

In [ ]:
if files:
    F_train = extract_feature_matrix(Xtr, CFG.ar_order)
    F_test = extract_feature_matrix(Xte, CFG.ar_order)
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler().fit(F_train)
    lda = fit_lda(scaler.transform(F_train), ytr)

    from sklearn.metrics import f1_score
    lda_pred = lda.predict(scaler.transform(F_test))
    lda_f1 = f1_score(yte, lda_pred, average="macro")

    import pickle
    lda_size_kb = len(pickle.dumps(lda)) / 1024.0

    x_bench = scaler.transform(F_test[:1])
    for _ in range(20):
        lda.predict(x_bench)
    times = []
    for _ in range(1000):
        t0 = time.perf_counter()
        lda.predict(x_bench)
        times.append((time.perf_counter() - t0) * 1000)

    results.append({"config": "lda_features_baseline", "macro_f1": lda_f1, "size_kb": lda_size_kb,
                     "median_ms": float(np.median(times)), "p95_ms": float(np.percentile(times, 95))})
    display(pd.DataFrame(results))


## 12. Pareto plots + the knee

In [ ]:
if files:
    results_df = pd.DataFrame(results)
    results_df.to_csv("project5_results.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(results_df["size_kb"], results_df["macro_f1"])
    for _, r in results_df.iterrows():
        axes[0].annotate(r["config"], (r["size_kb"], r["macro_f1"]), fontsize=8, rotation=20)
    axes[0].set_xlabel("model size (KB)")
    axes[0].set_ylabel("macro F1")
    axes[0].set_xscale("log")
    axes[0].set_title("Accuracy vs. size")

    axes[1].scatter(results_df["median_ms"], results_df["macro_f1"])
    for _, r in results_df.iterrows():
        axes[1].annotate(r["config"], (r["median_ms"], r["macro_f1"]), fontsize=8, rotation=20)
    axes[1].set_xlabel("median latency (ms), single CPU thread")
    axes[1].set_ylabel("macro F1")
    axes[1].set_title("Accuracy vs. latency")

    plt.tight_layout()
    plt.savefig("project5_pareto.png", dpi=150)
    plt.show()

    display(results_df.sort_values("macro_f1", ascending=False))


## 13. Per-layer sensitivity analysis

Quantize one Conv1d layer at a time (rest stay FP32) to see which layers are load-bearing for accuracy.

In [ ]:
def quantize_single_layer(model, layer_name):
    """Dynamic-quantize just one named Conv1d by monkey-patching its forward to
    round-trip through int8 fake-quant, leaving every other layer untouched."""
    m = copy.deepcopy(model)
    target = dict(m.named_modules())[layer_name]

    fq = tq.FakeQuantize.with_args(
        observer=tq.MovingAverageMinMaxObserver, quant_min=-128, quant_max=127, dtype=torch.qint8
    )()
    orig_forward = target.forward

    def patched_forward(x, _orig=orig_forward, _fq=fq):
        out = _orig(x)
        return _fq(out)

    target.forward = patched_forward
    return m


if files:
    conv_layers = [name for name, mod in fp32_model.named_modules() if isinstance(mod, nn.Conv1d)]
    sensitivity_rows = []
    for layer_name in conv_layers:
        m = quantize_single_layer(fp32_model, layer_name)
        f1 = eval_macro_f1(m, Xte, yte)
        sensitivity_rows.append({"layer": layer_name, "macro_f1_with_layer_quantized": f1,
                                  "delta_vs_fp32": f1 - results_df.query("config == 'fp32_baseline'")["macro_f1"].iloc[0]})
    sensitivity_df = pd.DataFrame(sensitivity_rows).sort_values("delta_vs_fp32")
    display(sensitivity_df)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(sensitivity_df["layer"], sensitivity_df["delta_vs_fp32"])
    ax.set_xlabel("macro F1 delta vs. FP32 (more negative = more sensitive)")
    ax.set_title("Per-layer quantization sensitivity")
    plt.tight_layout()
    plt.savefig("project5_layer_sensitivity.png", dpi=150)
    plt.show()


## 14. ONNX export + C++ harness (free C/C++ credential)

In [ ]:
if files and HAS_ONNX:
    dummy = torch.randn(1, CFG.n_channels, CFG.window_len)
    onnx_path = "emg_cnn_int8.onnx"
    torch.onnx.export(
        fp32_model, dummy, onnx_path,
        input_names=["emg_window"], output_names=["logits"],
        dynamic_axes={"emg_window": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=13,
    )
    onnx.checker.check_model(onnx.load(onnx_path))

    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    sample = Xte[:1].transpose(0, 2, 1).astype(np.float32)
    onnx_out = sess.run(None, {"emg_window": sample})[0]

    with torch.no_grad():
        torch_out = fp32_model(torch.tensor(sample)).numpy()

    max_abs_diff = np.max(np.abs(onnx_out - torch_out))
    print(f"ONNX vs. PyTorch max abs logit diff: {max_abs_diff:.2e}")
    assert max_abs_diff < 1e-3, "ONNX export diverges from PyTorch — investigate before shipping"
    print(f"Exported {onnx_path} — ready for the ONNX Runtime C++ API or int8 quantization via "
          f"onnxruntime.quantization.quantize_static.")
else:
    print("Install onnx + onnxruntime to run this section.")


```cpp
// minimal_onnx_harness.cpp — sketch for the C/C++ credential.
// Build against the ONNX Runtime C++ API (onnxruntime-cpp), link libonnxruntime.
#include <onnxruntime_cxx_api.h>
#include <vector>
#include <iostream>

int main() {
    Ort::Env env(ORT_LOGGING_LEVEL_WARNING, "emg_cnn");
    Ort::SessionOptions opts;
    opts.SetIntraOpNumThreads(1);  // match the single-thread microcontroller assumption
    Ort::Session session(env, L"emg_cnn_int8.onnx", opts);

    // n_channels x window_len, matches Config in the notebook
    std::vector<int64_t> input_shape = {1, 12, 400};
    std::vector<float> input_tensor(12 * 400, 0.0f);  // fill from a real window

    Ort::MemoryInfo mem_info = Ort::MemoryInfo::CreateCpu(OrtArenaAllocator, OrtMemTypeDefault);
    Ort::Value input = Ort::Value::CreateTensor<float>(
        mem_info, input_tensor.data(), input_tensor.size(), input_shape.data(), input_shape.size());

    const char* input_names[] = {"emg_window"};
    const char* output_names[] = {"logits"};
    auto output = session.Run(Ort::RunOptions{nullptr}, input_names, &input, 1, output_names, 1);

    float* logits = output.front().GetTensorMutableData<float>();
    // argmax logits here, compare against the Python reference in section 14 within tolerance.
    return 0;
}
```


## Notes / expected findings

- Watch for **LDA-on-features landing within 2–3 macro-F1 points of the FP32 CNN** at a fraction of the size/latency — if that's what the Pareto plot shows, that's the headline, not a footnote.
- FP16 on CPU is frequently **slower** than FP32 because most CPUs lack native FP16 arithmetic kernels — a legitimate result to report, not a bug to hide.
- Dynamic int8 only quantizes `nn.Linear` by default in PyTorch; this CNN's Conv1d layers stay FP32 under that path, so static/QAT should show a bigger win — that's expected, not an error.
- The per-layer sensitivity plot typically shows early conv layers are more quantization-tolerant than layers close to the classification head; if a layer's contribution is large and negative, that's the one to keep in float in any real int8 deployment.
